In [ ]:

# ============================================================
# CELL 1: Imports and global config
# ============================================================

from pathlib import Path
import gc
import time

import pandas as pd
import soundfile as sf
import torch
from tqdm.auto import tqdm

from transformers import (
    AutoProcessor,
    Qwen2AudioForConditionalGeneration,
    BitsAndBytesConfig,
)

from comet import download_model, load_from_checkpoint


PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
OUT_DIR = PROJECT_ROOT / "outputs" / "qwen2audio_eval_only_en_zh"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
DEVICE = "cuda:0"
PROMPT_TEXT = "Translate this English speech into Chinese."
MAX_NEW_TOKENS = 64
COMET_BATCH_SIZE = 16

DEV_MANIFEST = PROJECT_ROOT / "data" / "manifests" / "acl6060_dev_en_zh.csv"
EVAL_MANIFEST = PROJECT_ROOT / "data" / "manifests" / "acl6060_eval_en_zh.csv"

# This notebook does not fine-tune the model.
# Interpretation of "train on dev, eval on eval":
# - keep DEV separate for development / optional smoke tests
# - compute the reported scores only on EVAL

print("project root exists:", PROJECT_ROOT.exists())
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


# ============================================================
# CELL 2: Load DEV and EVAL separately
# ============================================================

dev_df = pd.read_csv(DEV_MANIFEST).copy()
eval_df = pd.read_csv(EVAL_MANIFEST).copy()

dev_df["audio_path"] = dev_df["audio_path"].apply(lambda p: str((PROJECT_ROOT / p).resolve()))
eval_df["audio_path"] = eval_df["audio_path"].apply(lambda p: str((PROJECT_ROOT / p).resolve()))

processor = AutoProcessor.from_pretrained(MODEL_ID)

comet_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_path)

print("dev rows:", len(dev_df))
print("eval rows:", len(eval_df))
print(dev_df.head(2))
print(eval_df.head(2))

# Optional:
# use DEV only for quick debugging / development smoke tests
DEV_SMOKE_N = 16
dev_smoke_df = dev_df.head(DEV_SMOKE_N).copy()


# ============================================================
# CELL 3: Helper functions only
# ============================================================

def generate_translation(model, processor, audio_path, prompt_text=PROMPT_TEXT, max_new_tokens=MAX_NEW_TOKENS):
    audio, sr = sf.read(str(audio_path))

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": str(audio_path)},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]

    text = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = processor(
        text=text,
        audios=[audio],
        sampling_rate=sr,
        return_tensors="pt",
    )

    inputs = {
        k: v.to(DEVICE) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )

    generated_ids = generated_ids[:, inputs["input_ids"].size(1):]

    pred = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    return pred


def run_inference(model, df_input, run_name):
    preds = []
    start = time.time()

    for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc=run_name):
        pred = generate_translation(model, processor, Path(row["audio_path"]))
        preds.append(pred)

    elapsed = time.time() - start

    out_df = df_input.copy()
    out_df["prediction"] = preds
    return out_df, elapsed


def compute_comet(pred_df, comet_model, batch_size=COMET_BATCH_SIZE):
    comet_data = [
        {"src": r["source_en"], "mt": r["prediction"], "ref": r["target_zh"]}
        for _, r in pred_df.iterrows()
    ]

    comet_out = comet_model.predict(comet_data, batch_size=batch_size, gpus=1)
    comet_score = comet_out.system_score if hasattr(comet_out, "system_score") else comet_out[1]
    return comet_score


def cleanup_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()



# ============================================================
# CELL 5: Baseline run on EVAL only + COMET + cleanup
# ============================================================

baseline_model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
baseline_model.eval()

baseline_eval_df, baseline_eval_time = run_inference(
    baseline_model,
    eval_df,
    "baseline_fp16_eval",
)

baseline_eval_out = OUT_DIR / "baseline_eval_preds.csv"
baseline_eval_df.to_csv(baseline_eval_out, index=False, encoding="utf-8")

baseline_eval_comet = compute_comet(baseline_eval_df, comet_model)

print("baseline eval predictions saved to:", baseline_eval_out)
print("baseline eval seconds:", round(baseline_eval_time, 2))
print("baseline eval COMET:", baseline_eval_comet)

cleanup_model(baseline_model)


# ============================================================
# CELL 6: Int8 run on EVAL only + COMET + cleanup
# ============================================================

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

quant_model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
quant_model.eval()

quant_eval_df, quant_eval_time = run_inference(
    quant_model,
    eval_df,
    "quantized_int8_eval",
)

quant_eval_out = OUT_DIR / "quant8_eval_preds.csv"
quant_eval_df.to_csv(quant_eval_out, index=False, encoding="utf-8")

quant_eval_comet = compute_comet(quant_eval_df, comet_model)

print("int8 eval predictions saved to:", quant_eval_out)
print("int8 eval seconds:", round(quant_eval_time, 2))
print("int8 eval COMET:", quant_eval_comet)

cleanup_model(quant_model)


# ============================================================
# CELL 7: Final summary on EVAL only
# ============================================================

summary = pd.DataFrame(
    [
        {
            "run": "baseline_fp16_eval",
            "split": "eval",
            "rows": len(baseline_eval_df),
            "seconds": baseline_eval_time,
            "comet": baseline_eval_comet,
        },
        {
            "run": "quantized_int8_eval",
            "split": "eval",
            "rows": len(quant_eval_df),
            "seconds": quant_eval_time,
            "comet": quant_eval_comet,
        },
    ]
)

baseline_score = summary.loc[summary["run"] == "baseline_fp16_eval", "comet"].iloc[0]
summary["comet_diff_vs_baseline"] = summary["comet"] - baseline_score
summary["speedup_vs_baseline"] = baseline_eval_time / summary["seconds"]

summary_out = OUT_DIR / "summary_eval_only.csv"
summary.to_csv(summary_out, index=False)

print("summary saved to:", summary_out)
print(summary)


In [2]:
print("summary saved to:", summary_out)
print(summary)


summary saved to: /home/paperspace/projects/iwslt2026-compression/outputs/qwen2audio_eval_only_en_zh/summary_eval_only.csv
                   run split  rows      seconds     comet  \
0   baseline_fp16_eval  eval   416   272.508722  0.779163   
1  quantized_int8_eval  eval   416  1048.651373  0.775192   

   comet_diff_vs_baseline  speedup_vs_baseline  
0                0.000000             1.000000  
1               -0.003971             0.259866  


In [3]:

# ============================================================
# CELL 4: Optional development smoke test on DEV only
# Uncomment if you want a quick sanity check before EVAL
# ============================================================

baseline_model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
baseline_model.eval()

dev_smoke_preds, dev_smoke_time = run_inference(
    baseline_model,
    dev_smoke_df,
    "baseline_fp16_dev_smoke",
)

dev_smoke_out = OUT_DIR / "baseline_dev_smoke_preds.csv"
dev_smoke_preds.to_csv(dev_smoke_out, index=False, encoding="utf-8")

dev_smoke_comet = compute_comet(dev_smoke_preds, comet_model)

print("dev smoke predictions saved to:", dev_smoke_out)
print("dev smoke seconds:", round(dev_smoke_time, 2))
print("dev smoke COMET:", dev_smoke_comet)

cleanup_model(baseline_model)


We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

baseline_fp16_dev_smoke:   0%|          | 0/16 [00:00<?, ?it/s]

/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.5` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
GP

dev smoke predictions saved to: /home/paperspace/projects/iwslt2026-compression/outputs/qwen2audio_eval_only_en_zh/baseline_dev_smoke_preds.csv
dev smoke seconds: 10.86
dev smoke COMET: 0.7878690734505653
